# Monetizing Quantum Compute with x402 HTTP Payments

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ionq-samples/getting-started/blob/main/x402_quantum_payments.ipynb)

The [x402 protocol](https://www.x402.org/) brings the HTTP `402 Payment Required` status code to life — letting any API gate access behind stablecoin micropayments, with no subscription or billing account needed.

This notebook shows how quantum compute providers can offer **pay-per-circuit** access to their APIs, and how AI agents can consume quantum resources autonomously.

## Why This Matters for Quantum

Quantum compute has a natural fit with micropayments:
- **Per-shot pricing** already mirrors per-request billing
- **AI agents** need autonomous compute access without human billing setup
- **Low-friction onboarding** — a new user can run a circuit with just a wallet, no account creation
- **Cross-border access** — USDC payments work globally without merchant accounts

## How x402 Works

```
1. Client sends request to API
2. Server returns 402 Payment Required + price quote
3. Client's wallet signs a USDC payment authorization
4. Client resends request with payment proof in headers
5. Server verifies payment on-chain, executes request, returns result
```

The protocol is open — any API can implement it. Payment settles on Base, Ethereum, or other EVM chains in USDC.

In [ ]:
# Install dependencies
!pip install requests qiskit qiskit-ionq

In [ ]:
import os
import json
import requests
from datetime import datetime

# ── Configuration ─────────────────────────────────────────
#
# x402 gateways act as payment middleware between clients
# and upstream APIs. You can:
#   1. Self-host a gateway (see github.com/coinbase/x402)
#   2. Use a hosted gateway like gateway.spraay.app
#
# The protocol is the same either way — the client just
# needs the gateway URL and a funded wallet.

IONQ_API_KEY = os.environ.get("IONQ_API_KEY", "your_ionq_api_key")

# Any x402-compatible gateway works here
X402_GATEWAY = os.environ.get(
    "X402_GATEWAY_URL",
    "https://gateway.spraay.app"  # default: Spraay's hosted gateway
)

print(f"Gateway: {X402_GATEWAY}")
print(f"Protocol: x402 (HTTP 402 Payment Required)")
print(f"Settlement: USDC on Base L2")

## Step 1: Discover Available Endpoints

x402 gateways publish a manifest at `/.well-known/x402-manifest` listing all payment-gated endpoints and their per-request prices. This lets agents discover capabilities and costs programmatically before committing funds.

In [ ]:
def discover_x402_endpoints(gateway_url, category=None):
    """
    Query an x402 gateway's manifest for available endpoints.
    
    This is a standard part of the x402 protocol — any compliant
    gateway exposes this endpoint for agent discovery.
    """
    response = requests.get(
        f"{gateway_url}/.well-known/x402-manifest",
        headers={"Accept": "application/json"}
    )
    
    if response.status_code == 200:
        manifest = response.json()
        endpoints = manifest.get("endpoints", [])
        if category:
            endpoints = [
                ep for ep in endpoints
                if category.lower() in ep.get("path", "").lower()
                or category.lower() in ep.get("description", "").lower()
            ]
        return endpoints
    return []

print("Querying x402 manifest for compute/inference endpoints...\n")
endpoints = discover_x402_endpoints(X402_GATEWAY, category="inference")
for ep in endpoints[:5]:
    print(f"  {ep.get('method', 'POST')} {ep.get('path', '')}")
    print(f"    Price: {ep.get('price', 'N/A')} USDC")
    print(f"    {ep.get('description', '')}")
    print()

## Step 2: Submit a Quantum Circuit with x402 Payment

The core pattern: send your normal IonQ API request through the gateway with an `X-PAYMENT` header. The gateway handles settlement and proxies the authenticated request upstream.

In [ ]:
def submit_circuit_x402(gateway_url, circuit_json, shots=100, api_key=None):
    """
    Submit a quantum circuit through an x402 payment gateway.
    
    This demonstrates the standard x402 client flow:
    
    1. POST with X-PAYMENT headers to signal willingness to pay
    2. If 402 → extract price quote, sign payment, retry
    3. If 200 → job submitted successfully, payment settled
    
    In production, wallet SDKs (e.g. Coinbase AgentKit, 
    Spraay Agent Wallet) automate the signing step.
    """
    payload = {
        "target": "simulator",  # or 'qpu.harmony' / 'qpu.aria-1'
        "shots": shots,
        "input": circuit_json
    }
    
    headers = {
        "Content-Type": "application/json",
        # x402 protocol headers
        "X-PAYMENT": "x402-usdc-base",
        "X-PAYMENT-NETWORK": "base",
    }
    if api_key:
        headers["Authorization"] = f"apiKey {api_key}"
    
    response = requests.post(
        f"{gateway_url}/v0.1/ai-inference",
        json=payload,
        headers=headers
    )
    
    if response.status_code == 402:
        # Standard x402 quote response
        quote = response.json()
        print("── x402 Payment Quote ──────────────────────")
        print(f"  Amount:  {quote.get('price', 'N/A')} USDC")
        print(f"  Network: {quote.get('network', 'base')}")
        print(f"  Pay to:  {quote.get('payTo', 'N/A')}")
        print(f"  Expires: {quote.get('expiry', 'N/A')}")
        print("────────────────────────────────────────────")
        print("  → In production, the wallet SDK auto-signs here")
        return {"status": "payment_required", "quote": quote}
    
    elif response.status_code in (200, 201):
        result = response.json()
        print(f"✅ Job submitted — ID: {result.get('id', 'N/A')}")
        print(f"   Status: {result.get('status', 'N/A')}")
        print(f"   Payment settled on-chain ✓")
        return result
    
    else:
        print(f"Error {response.status_code}: {response.text[:200]}")
        return None


# ── Bell State circuit ────────────────────────────────────
bell_state = {
    "format": "ionq.circuit.v0",
    "qubits": 2,
    "circuit": [
        {"gate": "h", "target": 0},
        {"gate": "cnot", "control": 0, "target": 1}
    ]
}

print("Circuit: H(q0) → CNOT(q0, q1)")
print("Expected: |00⟩ and |11⟩ with ~50/50 probability")
print()
result = submit_circuit_x402(X402_GATEWAY, bell_state, shots=1000, api_key=IONQ_API_KEY)

## Step 3: Qiskit Integration

The x402 layer sits transparently between Qiskit and IonQ — you build circuits normally, then route the submission through the payment gateway.

In [ ]:
from qiskit import QuantumCircuit

def qiskit_to_ionq_json(qc):
    """Convert a Qiskit QuantumCircuit to IonQ native JSON."""
    gate_map = {
        'h':  lambda q, _: {"gate": "h", "target": q[0]},
        'x':  lambda q, _: {"gate": "x", "target": q[0]},
        'y':  lambda q, _: {"gate": "y", "target": q[0]},
        'z':  lambda q, _: {"gate": "z", "target": q[0]},
        'cx': lambda q, _: {"gate": "cnot", "control": q[0], "target": q[1]},
        'rz': lambda q, p: {"gate": "rz", "target": q[0], "rotation": p[0]},
        'ry': lambda q, p: {"gate": "ry", "target": q[0], "rotation": p[0]},
    }
    gates = []
    for inst in qc.data:
        name = inst.operation.name
        if name in gate_map:
            qubits = [qc.find_bit(q).index for q in inst.qubits]
            gates.append(gate_map[name](qubits, inst.operation.params))
    return {"format": "ionq.circuit.v0", "qubits": qc.num_qubits, "circuit": gates}

# Build a 3-qubit GHZ state
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)

print(ghz.draw(output='text'))
print()

ionq_payload = qiskit_to_ionq_json(ghz)
print("Submitting via x402...")
result = submit_circuit_x402(X402_GATEWAY, ionq_payload, shots=1000)

## Implementing x402 on Your Own API

If you're a quantum compute provider and want to add x402 support, here's the minimal server-side pattern:

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/v1/jobs', methods=['POST'])
def submit_job():
    payment = request.headers.get('X-PAYMENT')
    
    if not payment:
        # Standard API key auth — existing flow unchanged
        return process_job_normally(request)
    
    payment_proof = request.headers.get('X-PAYMENT-PROOF')
    if not payment_proof:
        # Return 402 with price quote
        return jsonify({
            'price': '0.01',
            'currency': 'USDC',
            'network': 'base',
            'payTo': '0xYourAddress',
            'expiry': 300,
        }), 402
    
    # Verify payment on-chain, then process
    if verify_payment(payment_proof):
        return process_job_normally(request)
    else:
        return jsonify({'error': 'payment_invalid'}), 402
```

The x402 protocol is additive — it doesn't replace existing auth, it runs alongside it.

## Resources

- [x402 Protocol Specification](https://www.x402.org/) — the open standard
- [Coinbase x402 Reference Implementation](https://github.com/coinbase/x402) — server + client libraries
- [Spraay x402 Gateway](https://gateway.spraay.app) — hosted gateway with 115+ endpoints
- [IonQ API Reference](https://docs.ionq.com/api-reference/v0.2/)